In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from scipy.special import gamma

data = load_breast_cancer()
X = data.data
y = data.target
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

def fitness_function(position, X_train, X_test, y_train, y_test, alpha=0.99):
    selected_features = (position > 0.5).astype(int)
    num_selected = np.sum(selected_features)

    if num_selected == 0:
        return 0.0

    X_tr_subset = X_train[:, selected_features == 1]
    X_te_subset = X_test[:, selected_features == 1]

    clf = KNeighborsClassifier(n_neighbors=5)
    clf.fit(X_tr_subset, y_train)
    accuracy = clf.score(X_te_subset, y_test)

    num_total = X_train.shape[1]
    fitness = alpha * accuracy + (1 - alpha) * (1 - (num_selected / num_total))
    return fitness

def levy_flight(Lambda, size):
    temp = gamma(1 + Lambda) * np.sin(np.pi * Lambda / 2) / (gamma((1 + Lambda) / 2) * Lambda * 2**((Lambda - 1) / 2))
    sigma_u = temp**(1 / Lambda)
    sigma_v = 1
    u = np.random.normal(0, sigma_u, size)
    v = np.random.normal(0, sigma_v, size)
    step = u / (np.abs(v)**(1 / Lambda))
    return step

def cuckoo_search_feature_selection(n_nest=20, max_iter=30, pa=0.25, beta=1.5):
    n_features = X.shape[1]

    nests = np.random.rand(n_nest, n_features)
    fitness = np.zeros(n_nest)

    for i in range(n_nest):
        fitness[i] = fitness_function(nests[i], X_train, X_test, y_train, y_test)

    best_nest_index = np.argmax(fitness)
    best_nest = nests[best_nest_index].copy()
    best_fitness = fitness[best_nest_index]

    for iteration in range(max_iter):
        new_nests = nests.copy()
        for i in range(n_nest):
            step_size = levy_flight(beta, n_features)
            step_direction = nests[i] - best_nest
            new_nests[i] = nests[i] + 0.01 * step_size * step_direction
            new_nests[i] = np.clip(new_nests[i], 0, 1)

            new_fit = fitness_function(new_nests[i], X_train, X_test, y_train, y_test)
            if new_fit > fitness[i]:
                nests[i] = new_nests[i]
                fitness[i] = new_fit

        for i in range(n_nest):
            if np.random.rand() < pa:
                random_nest_1 = nests[np.random.randint(0, n_nest)]
                random_nest_2 = nests[np.random.randint(0, n_nest)]
                step_size = np.random.rand() * (random_nest_1 - random_nest_2)
                nests[i] = np.clip(nests[i] + step_size, 0, 1)
                fitness[i] = fitness_function(nests[i], X_train, X_test, y_train, y_test)

        current_best_idx = np.argmax(fitness)
        if fitness[current_best_idx] > best_fitness:
            best_fitness = fitness[current_best_idx]
            best_nest = nests[current_best_idx].copy()

        print(f"Iteration {iteration+1}/{max_iter} - Best Fitness: {best_fitness:.4f}")

    return best_nest, best_fitness

best_solution, best_score = cuckoo_search_feature_selection(n_nest=15, max_iter=20)

selected_features_indices = np.where(best_solution > 0.5)[0]
selected_features_names = feature_names[selected_features_indices]

print("\n--- Feature Selection Results ---")
print(f"Total features available: {len(feature_names)}")
print(f"Selected features count: {len(selected_features_names)}")
print(f"Selected features list: {list(selected_features_names)}")

Iteration 1/20 - Best Fitness: 0.9494
Iteration 2/20 - Best Fitness: 0.9494
Iteration 3/20 - Best Fitness: 0.9494
Iteration 4/20 - Best Fitness: 0.9494
Iteration 5/20 - Best Fitness: 0.9494
Iteration 6/20 - Best Fitness: 0.9494
Iteration 7/20 - Best Fitness: 0.9494
Iteration 8/20 - Best Fitness: 0.9494
Iteration 9/20 - Best Fitness: 0.9494
Iteration 10/20 - Best Fitness: 0.9494
Iteration 11/20 - Best Fitness: 0.9494
Iteration 12/20 - Best Fitness: 0.9494
Iteration 13/20 - Best Fitness: 0.9494
Iteration 14/20 - Best Fitness: 0.9494
Iteration 15/20 - Best Fitness: 0.9494
Iteration 16/20 - Best Fitness: 0.9494
Iteration 17/20 - Best Fitness: 0.9494
Iteration 18/20 - Best Fitness: 0.9494
Iteration 19/20 - Best Fitness: 0.9494
Iteration 20/20 - Best Fitness: 0.9494

--- Feature Selection Results ---
Total features available: 30
Selected features count: 13
Selected features list: [np.str_('mean radius'), np.str_('mean perimeter'), np.str_('mean compactness'), np.str_('mean concavity'), np.st